# OPT-30B (and other big models) via the OFFLOAD path

Runs the **block-streaming fake-quant accuracy** pipeline (`iso_energy_125m.py`, `OFFLOAD=1`) — the right tool for models bigger than the GPU. The model lives in CPU RAM and one decoder block at a time streams to the GPU for quantization/eval.

**Not** the CUDA-kernel notebook: this does not use the hand-written `.cu` kernels. It's the `bit_split` accuracy path, and it evaluates the **full WikiText-2 test set** (directly comparable to published PPL).

**Your box:** A100 40 GB GPU, 83.5 GB RAM, ~189 GB free disk. OPT-30B fp16 ≈ 60 GB → lives in CPU RAM (fits 83.5 GB), streams to the 40 GB GPU one block at a time. The ~60 GB download goes to disk.

**Windows:** calibration is `NCALIB=16` samples (default; sweep shows accuracy plateaus by ~8). Eval is the whole test set automatically — no knob.

## 0. Runtime check (confirm GPU / RAM / disk)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import psutil, shutil
print(f"RAM total : {psutil.virtual_memory().total/1e9:.1f} GB")
print(f"Disk free : {shutil.disk_usage('/').free/1e9:.1f} GB   (opt-30b download needs ~60 GB)")

## 1. Clone the repo

Private repo → add a GitHub PAT as a Colab Secret named `GH_TOKEN` (🔑 sidebar). Public → leave it unset.

In [ ]:
import os
tok = ''
try:
    from google.colab import userdata
    tok = userdata.get('GH_TOKEN')
except Exception:
    tok = os.environ.get('GH_TOKEN', '')
os.environ['REPO_URL'] = f"https://{tok + '@' if tok else ''}github.com/jvap2/Thesis_Compression.git"
!rm -rf /content/Thesis_Compression
!git clone --depth 1 "$REPO_URL" /content/Thesis_Compression
%cd /content/Thesis_Compression/Python_Jenks_Test/Jenks_Tests
!ls iso_energy_125m.py FP_Quantization_Experiments/bit_split.py

In [ ]:
!pip -q install "transformers==4.46.3" datasets accelerate

## 2. Environment for the offload run

| var | value | why |
|---|---|---|
| `OFFLOAD` | `1` | quantize by streaming one decoder block to the GPU (model stays in CPU RAM) |
| `EVAL_OFFLOAD` | `auto` | 30B (60 GB) > 40 GB GPU → eval streams automatically |
| `NCALIB` | `16` | calibration samples (default; accuracy plateaus by ~8, matches the other offload rows) |
| `RETAIN` | `1` / `0` | 1 = main (outlier layers kept FP16), 0 = ablation (collapse evidence) — set per-run below |

Eval sequence length is fixed at 2048 over the full WikiText-2 test set.

In [ ]:
import os
os.environ['OFFLOAD'] = '1'
os.environ['EVAL_OFFLOAD'] = 'auto'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['NCALIB'] = '16'            # calibration samples; set '8' to ~halve calib time (negligible accuracy change)
# os.environ['CALIB_SEQLEN'] = '1024'  # uncomment ONLY if calibration OOMs
# Gated (Llama) models need a token:
# from google.colab import userdata; os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('env set:', {k: os.environ[k] for k in ['OFFLOAD','EVAL_OFFLOAD','NCALIB']})

## 3. Smoke test first (cheap) — validate the offload path before the 60 GB download

`opt-1.3b` runs in a couple of minutes and confirms the environment, clone, and offload code all work end-to-end. Don't skip this before spending A100 time on 30B.

In [ ]:
!python3 -u iso_energy_125m.py facebook/opt-1.3b

## 4. OPT-30B

First run **downloads ~60 GB** to disk (10–20 min) and holds the model in ~60 GB RAM. Expect the whole thing (download + calibrate + full-test-set streaming eval) to take a while — likely **30–90 min per run** on a streamed 40 GB card.

Watch for the `eval: model=..GiB gpu=..GiB -> STREAMING offload` line confirming it picked the streaming path. Each run appends a row to `iso_results.csv`.

In [ ]:
# Main run (RETAIN defaults to 1 = outlier layers kept FP16 — the headline W4A4 number)
!python3 -u iso_energy_125m.py facebook/opt-30b

In [ ]:
# Ablation run (RETAIN=0 = no outlier retention — the collapse-evidence row)
!RETAIN=0 python3 -u iso_energy_125m.py facebook/opt-30b

## 5. Results

`iso_results.csv` accumulates every run's row. Download it to fold into the paper table.

In [ ]:
import pandas as pd
df = pd.read_csv('iso_results.csv')
print(df.tail(8).to_string())
from google.colab import files; files.download('iso_results.csv')